In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_json_path = 'rca100/cases/t093/task.json'
with open(task_json_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from the task.json
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is related to the "cart" service operation `POST /oteldemo.CartService/GetCart`. The issue is described as a "cart interface traffic drop alert" and occurred within the time window from `2026-04-04 23:29:47.051744` to `2026-04-04 23:38:05` (UTC+8). The user is requesting an analysis of the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

('', {'start': '2026-04-04T23:29:47.051744+08:00', 'end': '2026-04-04T23:38:05+08:00'}, {'entity_id': '7bb1b1512922edfdefac66f31c640a81', 'entity_name': 'cart::POST /oteldemo.CartService/GetCart', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">cart接口流量下跌告警<alert_event event_id="9140b17c9e824350ae2401fbf1a16ff9" trans_id="c50qh705u13cd4kl71ga4q03f6" rule_id="51f70b52-a9a7-4135-b094-5bb8ba67d9ce" rule_name="cart接口流量下跌告警" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_path = 'rca100/cases/t093/metrics.parquet'
metrics_df = pd.read_parquet(metrics_path)

# List all unique KPIs available in the dataset
unique_kpis = metrics_df['metric'].unique()
unique_kpis

```
Out[2]:
```
The dataset contains various KPIs, including traffic-related ones such as `request_count`, `error_count`, `error_rate`, `latency`, and `slow`. These KPIs are relevant for analyzing the traffic drop issue in the "cart" service.

The original code execution output of IPython Kernel is also provided below for reference:

['node_ready_status' 'node_not_ready_status' 'node_memory_usage_rate'
 'node_pod_running_count' 'node_disk_usage_rate' 'node_cpu_usage_rate'
 'deployment_desired_replicas' 'deployment_available_replicas'
 'deployment_ready_replicas' 'deployment_availability_rate'
 'deployment_memory_usage_total' 'deployment_cpu_usage_total'
 'deployment_cpu_usage_vs_limits' 'deployment_cpu_usage_vs_requests'
 'apiserver_non_2xx_read_request_qps' 'cluster_failed_node_count'
 'cluster_number_of_not_ready_pods' 'cluster_node_count'
 'apiserver_concurrent_connections' 'cluster_number_of_pod'
 'apiserver_non_2xx_write_request_qps' 'apiserver_requests_per_second'
 'namespace_number_of_not_ready_pods' 'namespace_number_of_pods'
 'slow_count' 'error_count' 'request_count' 'avg_request_latency_seconds'
 'arms_jvm_gc_total' 'arms_jvm_gc_seconds_total' 'arms_jvm_gc_delta'
 'arms_jvm_gc_seconds_delta' 'arms_jvm_mem_used_bytes'
 'arms_jvm_mem_init_bytes' 'arms_jvm_mem_committed_bytes'
 'arms_jvm_mem_max_bytes' 'arms_jvm_threads_count'
 'arms_thread_pool_active_thread_count'
 'arms_thread_pool_current_thread_count'
 'arms_thread_pool_max_thread_count' 'arms_thread_pool_max_pool_size'
 'arms_thread_pool_core_pool_size' 'arms_thread_pool_scheduled_task_count'
 'arms_thread_pool_completed_task_count'
 'arms_thread_pool_rejected_task_count' 'arms_thread_pool_queue_size'
 'arms_grouped_thread_count' 'arms_grouped_thread_cpu_time'
 'arms_grouped_thread_runnable_count' 'arms_grouped_thread_waiting_count'
 'arms_grouped_thread_timed_waiting_count' 'cpu' 'latency' 'mem'
 'workload' 'error' 'slow' 'error_rate']```
```

In [ ]:
# In[3]:

import numpy as np
from pytz import timezone
from datetime import datetime

# Convert time column to numeric and filter for service-level components
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')
service_metrics = metrics_df[metrics_df['entity_set'] == 'apm.service.legacy']

# Define the relevant KPIs
relevant_kpis = ['request_count', 'error_count', 'error_rate', 'latency', 'slow']

# Calculate global P95 thresholds for each KPI
p95_thresholds = service_metrics[service_metrics['metric'].isin(relevant_kpis)] \
    .groupby('metric')['value'].quantile(0.95)

# Convert alert time window to timestamps
alert_start = datetime.fromisoformat('2026-04-04T23:29:47.051744+08:00').astimezone(timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-04T23:38:05+08:00').astimezone(timezone('Asia/Shanghai'))
alert_start_ts = int(alert_start.timestamp() * 1_000_000)  # Convert to microseconds
alert_end_ts = int(alert_end.timestamp() * 1_000_000)      # Convert to microseconds

# Filter data within the alert time window
filtered_metrics = service_metrics[
    (service_metrics['metric'].isin(relevant_kpis)) &
    (service_metrics['time'] >= alert_start_ts) &
    (service_metrics['time'] <= alert_end_ts)
]

p95_thresholds, filtered_metrics

```
Out[3]:
```
The global P95 thresholds for the relevant KPIs across the entire dataset are as follows: `latency` is 0.616176, and `slow` is 2636.8. Within the alert time window, metrics for various services, including `latency` and `slow`, were recorded. The "cart" service shows a `latency` of 0.003305 and a `slow` value of 85.505882, which are below the global P95 thresholds. Further analysis is needed to identify anomalies or root causes.

The original code execution output of IPython Kernel is also provided below for reference:

(metric
latency       0.616176
slow       2636.800000
Name: value, dtype: float64,                    time domain          entity_set entity_id      entity_name   metric      value metric_set_id service
88246  1775316617000000    apm  apm.service.legacy                 accounting  latency   0.024511          None    None
88250  1775316617000000    apm  apm.service.legacy                         ad  latency   0.000503          None    None
88255  1775316617000000    apm  apm.service.legacy                       cart  latency   0.003305          None    None
88257  1775316617000000    apm  apm.service.legacy                       cart     slow  85.505882          None    None
88260  1775316617000000    apm  apm.service.legacy                   checkout  latency   0.289242          None    None
...                 ...    ...                 ...       ...              ...      ...        ...           ...     ...
89410  1775317067000000    apm  apm.service.legacy                    payment  latency   0.000381          None    None
89414  1775317067000000    apm  apm.service.legacy            product-catalog  latency   0.000074          None    None
89418  1775317067000000    apm  apm.service.legacy                      quote  latency   0.000276          None    None
89422  1775317067000000    apm  apm.service.legacy             recommendation  latency   0.010888          None    None
89426  1775317067000000    apm  apm.service.legacy                   shipping  latency   0.001161          None    None

[320 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Filter the metrics data for the 'request_count' KPI within the alert time window
request_count_metrics = service_metrics[
    (service_metrics['metric'] == 'request_count') &
    (service_metrics['time'] >= alert_start_ts) &
    (service_metrics['time'] <= alert_end_ts)
]

# Check for anomalies or significant drops in the 'request_count' values for the 'cart' service and other services
request_count_summary = request_count_metrics.groupby('entity_name')['value'].agg(['min', 'max', 'mean', 'std'])

request_count_metrics, request_count_summary

```
Out[4]:
```
No `request_count` data is available for any service-level components, including the "cart" service, within the alert time window. This absence of data may indicate a potential issue with data collection or a complete drop in traffic during this period. Further investigation is needed to confirm the cause.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [time, domain, entity_set, entity_id, entity_name, metric, value, metric_set_id, service]
Index: [], Empty DataFrame
Columns: [min, max, mean, std]
Index: [])```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_path = 'rca100/cases/t093/traces.parquet'
traces_df = pd.read_parquet(traces_path)

# Convert time-related columns to numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')

# Filter traces within the alert time window and involving the 'cart' service
cart_traces = traces_df[
    (traces_df['startTime'] >= alert_start_ts * 1_000) &  # Convert microseconds to nanoseconds
    (traces_df['endTime'] <= alert_end_ts * 1_000) &
    (traces_df['serviceName'].str.contains('cart', case=False, na=False))
]

cart_traces

```
Out[5]:
```
The filtered traces within the alert time window include 6,666 entries involving the "cart" service. These traces cover operations such as `POST /oteldemo.CartService/GetCart` and `POST /oteldemo.CartService/AddItem`. Further analysis of these traces is needed to identify any errors, anomalies, or disruptions in the service call chain.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind                            spanName  ...  __topic__      __tag__:__pack_id__                                             events links    ip
398041  3c3592131e2bdb7d96db1d1266565ad0  713f69da87b089cb  5ce209c656118623    1                        GetInventory  ...             14AE0E1682356D2A-3CF04B                                               None  None  None
398042  3c3592131e2bdb7d96db1d1266565ad0  5ce209c656118623  f857d2a209f27177    2  POST /oteldemo.CartService/AddItem  ...             14AE0E1682356D2A-3CF04B                                               None  None  None
398043  91a84e0f3dd0a124223100a1445ed09e  9a38d8f16d68e888  cc7194ee7a29acea    1                        GetInventory  ...             14AE0E1682356D2A-3CF04B                                               None  None  None
398044  91a84e0f3dd0a124223100a1445ed09e  cc7194ee7a29acea  261c3c907e80cb51    2  POST /oteldemo.CartService/AddItem  ...             14AE0E1682356D2A-3CF04B                                               None  None  None
398045  91a84e0f3dd0a124223100a1445ed09e  d4becb408d740678  1901d3ddd6e6a427    2  POST /oteldemo.CartService/GetCart  ...             14AE0E1682356D2A-3CF04B  [{"attributes":{},"name":"Fetch cart","timesta...  None  None
...                                  ...               ...               ...  ...                                 ...  ...        ...                      ...                                                ...   ...   ...
446034  679cb0b2dbf350428a220282f6353389  53781cae96b83f4d  6910b942d8637e95    2  POST /oteldemo.CartService/GetCart  ...             D94F74C386BA7A8B-3CBE8F  [{"attributes":{},"name":"Fetch cart","timesta...  None  None
446035  a96b2eb62ad700ba2b6814baa4c02625  4b89899bd6918e3e  27e059bca29a6571    1                        GetInventory  ...             D94F74C386BA7A8B-3CBE8F                                               None  None  None
446503  c30a28ffab0e11b1f22c627dd0cbcb97  14c58d8790e4eba9  aa26bc90b229fc62    3                                 GET  ...             22D2A3D145F2EFD3-3C86B6                                               None  None  None
446504  c30a28ffab0e11b1f22c627dd0cbcb97  aa26bc90b229fc62  a3b75c3907dc070d    1                        GetInventory  ...             22D2A3D145F2EFD3-3C86B6                                               None  None  None
446505  c30a28ffab0e11b1f22c627dd0cbcb97  a3b75c3907dc070d  2d386e1424fc6e8f    2  POST /oteldemo.CartService/AddItem  ...             22D2A3D145F2EFD3-3C86B6  [{"attributes":{"exception.type":"System.Net.H...  None  None

[6666 rows x 21 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Filter traces involving the 'cart' service for non-zero statusCode or error-related statusMessage
cart_traces['statusCode'] = pd.to_numeric(cart_traces['statusCode'], errors='coerce')
error_traces = cart_traces[
    (cart_traces['statusCode'] != 0) | 
    (cart_traces['statusMessage'].str.contains('error', case=False, na=False))
]

error_traces

```
Out[6]:
```
Among the 6,666 traces involving the "cart" service, 3 traces were identified with errors or disruptions. These include operations such as `POST /oteldemo.CartService/AddItem` and others, with error-related details in the `statusMessage` or non-zero `statusCode`. These traces indicate potential issues in the service call chain during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind                            spanName  ...  __topic__      __tag__:__pack_id__                                             events links    ip
446503  c30a28ffab0e11b1f22c627dd0cbcb97  14c58d8790e4eba9  aa26bc90b229fc62    3                                 GET  ...             22D2A3D145F2EFD3-3C86B6                                               None  None  None
446504  c30a28ffab0e11b1f22c627dd0cbcb97  aa26bc90b229fc62  a3b75c3907dc070d    1                        GetInventory  ...             22D2A3D145F2EFD3-3C86B6                                               None  None  None
446505  c30a28ffab0e11b1f22c627dd0cbcb97  a3b75c3907dc070d  2d386e1424fc6e8f    2  POST /oteldemo.CartService/AddItem  ...             22D2A3D145F2EFD3-3C86B6  [{"attributes":{"exception.type":"System.Net.H...  None  None

[3 rows x 21 columns]```
```

In [ ]:
# In[7]:

# Analyze the 3 error traces to identify the call chain and the most downstream faulty service
error_trace_analysis = error_traces[['traceId', 'spanId', 'parentSpanId', 'spanName', 'serviceName', 'statusCode', 'statusMessage']]

# Group by traceId to analyze each trace separately and sort by parent-child relationships
error_trace_analysis_sorted = error_trace_analysis.sort_values(by=['traceId', 'parentSpanId', 'spanId'])

error_trace_analysis_sorted

```
Out[7]:
```
The analysis of the 3 error traces reveals that the "cart" service encountered a connection issue when trying to communicate with the "inventory" service (`Connection refused (inventory:9090)`). This indicates that the root cause of the fault lies in the "inventory" service, as it failed to respond to requests from the "cart" service, causing disruptions in the service call chain.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId                            spanName serviceName  statusCode                        statusMessage
446505  c30a28ffab0e11b1f22c627dd0cbcb97  a3b75c3907dc070d  2d386e1424fc6e8f  POST /oteldemo.CartService/AddItem        cart           2  Connection refused (inventory:9090)
446504  c30a28ffab0e11b1f22c627dd0cbcb97  aa26bc90b229fc62  a3b75c3907dc070d                        GetInventory        cart           2  Connection refused (inventory:9090)
446503  c30a28ffab0e11b1f22c627dd0cbcb97  14c58d8790e4eba9  aa26bc90b229fc62                                 GET        cart           2```
```